In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kimanaru/dataset-tien-xu-li/val_ml_clean_text.csv
/kaggle/input/datasets/kimanaru/dataset-tien-xu-li/train_ml_clean_text.csv
/kaggle/input/datasets/kimanaru/dataset-tien-xu-li/test_ml_clean_text.csv


In [2]:
import pandas as pd
from pathlib import Path

RESULT_DIR = Path("/kaggle/input/datasets/kimanaru/dataset-tien-xu-li")

train_df = pd.read_csv(RESULT_DIR / "train_ml_clean_text.csv")
val_df = pd.read_csv(RESULT_DIR / "val_ml_clean_text.csv")
test_df = pd.read_csv(RESULT_DIR / "test_ml_clean_text.csv")

train_df["ml_clean_text"] = train_df["ml_clean_text"].fillna("").astype(str)
val_df["ml_clean_text"] = val_df["ml_clean_text"].fillna("").astype(str)
test_df["ml_clean_text"] = test_df["ml_clean_text"].fillna("").astype(str)

train_df["category"] = train_df["category"].astype(str)
val_df["category"] = val_df["category"].astype(str)
test_df["category"] = test_df["category"].astype(str)

X_train = train_df["ml_clean_text"]
y_train = train_df["category"]

X_val = val_df["ml_clean_text"]
y_val = val_df["category"]

X_test = test_df["ml_clean_text"]
y_test = test_df["category"]

# Thử các model

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import f1_score, classification_report
import pandas as pd
import time
import joblib

from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB, ComplementNB

from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
import lightgbm as lgb

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, classification_report
import pandas as pd
import time
import joblib

BOW_MAX_FEATURES = 250000
BOW_NGRAM_RANGE = (1, 2)

RANDOM_STATE = 42
CLASS_WEIGHT = "balanced"
MAX_ITER = 5000

svc_C_list = [0.01, 0.03, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0, 3.0, 5.0]

results = []
best_score = -1
best_model = None
best_C = None

for i, C in enumerate(svc_C_list, 1):
    print("=" * 80)
    print(f"Running {i}/{len(svc_C_list)}")
    print("C:", C)

    model = Pipeline([
        ("bow", CountVectorizer(
            max_features=BOW_MAX_FEATURES,
            ngram_range=BOW_NGRAM_RANGE,
            lowercase=False
        )),
        ("svc", LinearSVC(
            C=C,
            class_weight=CLASS_WEIGHT,
            max_iter=MAX_ITER,
            random_state=RANDOM_STATE
        ))
    ])

    start = time.time()

    model.fit(X_train, y_train)

    y_val_pred = model.predict(X_val)
    val_macro_f1 = f1_score(y_val, y_val_pred, average="macro")

    elapsed = time.time() - start

    print("Validation Macro F1:", val_macro_f1)
    print("Time:", round(elapsed, 2), "seconds")

    results.append({
        "feature_type": "BoW",
        "max_features": BOW_MAX_FEATURES,
        "ngram_range": BOW_NGRAM_RANGE,
        "C": C,
        "class_weight": CLASS_WEIGHT,
        "max_iter": MAX_ITER,
        "random_state": RANDOM_STATE,
        "val_macro_f1": val_macro_f1,
        "time_seconds": elapsed
    })

    if val_macro_f1 > best_score:
        best_score = val_macro_f1
        best_model = model
        best_C = C
        print("New best SVC!")

results_df = pd.DataFrame(results).sort_values("val_macro_f1", ascending=False)

print("=" * 80)
print("BEST C:", best_C)
print("BEST VALIDATION MACRO F1:", best_score)

results_df

Running 1/10
C: 0.01
Validation Macro F1: 0.6882923509572217
Time: 16.64 seconds
New best SVC!
Running 2/10
C: 0.03
Validation Macro F1: 0.6904040174849074
Time: 19.46 seconds
New best SVC!
Running 3/10
C: 0.05
Validation Macro F1: 0.6885297075783134
Time: 21.55 seconds
Running 4/10
C: 0.1
Validation Macro F1: 0.6841285860505252
Time: 25.43 seconds
Running 5/10
C: 0.3
Validation Macro F1: 0.6756094785948636
Time: 38.52 seconds
Running 6/10
C: 0.5
Validation Macro F1: 0.6688737161132874
Time: 50.77 seconds
Running 7/10
C: 1.0
Validation Macro F1: 0.66344253262461
Time: 77.81 seconds
Running 8/10
C: 2.0
Validation Macro F1: 0.6577242389767802
Time: 121.26 seconds
Running 9/10
C: 3.0
Validation Macro F1: 0.6537866394050261
Time: 158.69 seconds
Running 10/10
C: 5.0


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Validation Macro F1: 0.6502971453065697
Time: 229.49 seconds
BEST C: 0.03
BEST VALIDATION MACRO F1: 0.6904040174849074


,feature_type,max_features,ngram_range,C,class_weight,max_iter,random_state,val_macro_f1,time_seconds
1,BoW,250000,"(1, 2)",0.03,balanced,5000,42,0.690404,19.464571
2,BoW,250000,"(1, 2)",0.05,balanced,5000,42,0.688530,21.553218
0,BoW,250000,"(1, 2)",0.01,balanced,5000,42,0.688292,16.639519
3,BoW,250000,"(1, 2)",0.10,balanced,5000,42,0.684129,25.429165
4,BoW,250000,"(1, 2)",0.30,balanced,5000,42,0.675609,38.519686
5,BoW,250000,"(1, 2)",0.50,balanced,5000,42,0.668874,50.770350
6,BoW,250000,"(1, 2)",1.00,balanced,5000,42,0.663443,77.814010
7,BoW,250000,"(1, 2)",2.00,balanced,5000,42,0.657724,121.257776
8,BoW,250000,"(1, 2)",3.00,balanced,5000,42,0.653787,158.691969
9,BoW,250000,"(1, 2)",5.00,balanced,5000,42,0.650297,229.493427


In [5]:
y_test_pred = best_model.predict(X_test)

print("Best C:", best_C)
print("Test Macro F1:", f1_score(y_test, y_test_pred, average="macro"))
print(classification_report(y_test, y_test_pred))

Best C: 0.03
Test Macro F1: 0.6997349841556185
                precision    recall  f1-score   support

  BLACK VOICES       0.58      0.58      0.58       687
      BUSINESS       0.62      0.66      0.64       899
        COMEDY       0.58      0.54      0.56       810
 ENTERTAINMENT       0.80      0.78      0.79      2605
  FOOD & DRINK       0.75      0.83      0.78       951
HEALTHY LIVING       0.43      0.41      0.42      1004
 HOME & LIVING       0.79      0.78      0.78       648
     PARENTING       0.62      0.67      0.65      1319
       PARENTS       0.41      0.37      0.39       593
      POLITICS       0.90      0.87      0.88      5341
  QUEER VOICES       0.82      0.78      0.80       952
        SPORTS       0.75      0.84      0.79       761
STYLE & BEAUTY       0.86      0.87      0.86      1472
        TRAVEL       0.82      0.83      0.82      1485
      WELLNESS       0.73      0.76      0.75      2692

      accuracy                           0.76     22219